
# 02. Teacher On Held-Out τ³-Bench Retail Eval

This notebook evaluates the teacher model on the same held-out τ³-Bench retail test split used in notebook 01. It does **not** create training data.

The agent under test is the vLLM-served teacher. The user simulator is configured in `.env` and served through the local ChatGPT subscription shim, exposed through LiteLLM as an OpenAI-compatible endpoint.

The output file is resume-safe. After each completed task, the result is written to disk. If the notebook stops, rerunning the eval cell skips cached task ids and continues from the missing ones.



## What Runs Where

```mermaid
flowchart TD
  A["τ³-Bench retail test task"] --> B["User simulator"]
  B --> C["local ChatGPT subscription shim"]
  C --> D["LiteLLM OpenAI-compatible call"]
  A --> E["Retail environment and tools"]
  F["Teacher agent"] --> G["Qwen prompt rendered locally"]
  G --> H["vLLM /v1/completions"]
  H --> I["Raw Qwen XML text"]
  I --> J["Shared Qwen tool parser"]
  J --> E
  E --> F
  B --> F
  E --> K["τ³-Bench evaluator reward"]
```

The key point: ChatGPT only plays the customer. It is not the teacher and it is not scored as the agent. The teacher agent is still the Qwen 35B model served by vLLM.


In [1]:
from __future__ import annotations

from collections import Counter
from typing import Any
import json

from pathlib import Path
import sys

cwd = Path.cwd()
ROOT = cwd if (cwd / "common" / "config.py").exists() else cwd.parent
if not (ROOT / "common" / "config.py").exists():
    raise RuntimeError("Run this notebook from the repo root or a blog folder under the repo root.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from common import config as cfg
from common import retail_eval
from common import tau_runtime
from common import teacher_backends
from common import user_simulator

paths = cfg.setup_notebook_paths(blog_dir_name="1-distilling-a-0-8b-tool-calling-agent")
ROOT = paths.root
BLOG_DIR = paths.blog_dir
DATA_DIR = paths.data_dir
OUTPUT_DIR = paths.output_dir
ENV_PATH = paths.env_path
DOTENV_LOADED = paths.dotenv_loaded

TAU_BENCH_REPO_REVISION = cfg.TAU_BENCH_REPO_REVISION
TAU_BENCH_DOMAIN = "retail"
TEACHER_MODEL = cfg.TEACHER_MODEL

print("Project root:", ROOT)
print("Blog dir:", BLOG_DIR)
print("Output dir:", OUTPUT_DIR)
print("Loaded .env:", DOTENV_LOADED, "from", ENV_PATH)
print("Benchmark: τ³-Bench", TAU_BENCH_DOMAIN)
print("Pinned tau2-bench revision:", TAU_BENCH_REPO_REVISION[:8])
print("Teacher model:", TEACHER_MODEL)

Project root: /Users/kargarisaac/codes/personal/distillation-blogs
Blog dir: /Users/kargarisaac/codes/personal/distillation-blogs/1-distilling-a-0-8b-tool-calling-agent
Output dir: /Users/kargarisaac/codes/personal/distillation-blogs/outputs
Loaded .env: True from /Users/kargarisaac/codes/personal/distillation-blogs/.env
Benchmark: τ³-Bench retail
Pinned tau2-bench revision: c42db6cc
Teacher model: mlx-community/Qwen3.5-35B-A3B-8bit



## Start The User Simulator Backend

This starts the local ChatGPT subscription shim inside the notebook process. τ³-Bench talks to the user simulator through LiteLLM, and LiteLLM points at the local shim.

The shim base URL has a random free port by default. That makes reruns less likely to collide with another process.


In [2]:

user_simulator_runtime = user_simulator.start_tau_bench_user_simulator_from_env(
    existing_shim=globals().get("chatgpt_user_simulator_shim")
)
chatgpt_user_simulator_shim = user_simulator_runtime.shim
TAU_BENCH_USER_SIMULATOR_LLM = user_simulator_runtime.model
TAU_BENCH_USER_SIMULATOR_ARGS = user_simulator_runtime.args

print("User simulator model:", TAU_BENCH_USER_SIMULATOR_LLM)
print("ChatGPT subscription shim model:", user_simulator_runtime.shim_model)
print("ChatGPT subscription shim base URL:", chatgpt_user_simulator_shim.base_url)
print("User simulator args:", user_simulator.public_user_simulator_args(TAU_BENCH_USER_SIMULATOR_ARGS))
print("User simulator API key present:", bool(TAU_BENCH_USER_SIMULATOR_ARGS.get("api_key")))


User simulator model: openai/gpt-5.4-mini
ChatGPT subscription shim model: gpt-5.4-mini
ChatGPT subscription shim base URL: http://127.0.0.1:50757/v1
User simulator args: {'api_base': 'http://127.0.0.1:50757/v1', 'temperature': 0.0, 'num_retries': 6, 'timeout': 300}
User simulator API key present: True



## Configure The vLLM Teacher

Start this server in another terminal if it is not already running:

```bash
source ~/.venv-vllm-metal/bin/activate
HF_HUB_DISABLE_XET=1 VLLM_METAL_MEMORY_FRACTION=0.95 vllm serve mlx-community/Qwen3.5-35B-A3B-8bit \
  --host 127.0.0.1 \
  --port 8092 \
  --max-model-len 81920 \
  --dtype float16 \
  --trust-remote-code \
  --enforce-eager \
  --generation-config vllm
```

The notebook sends raw Qwen prompts to `/v1/completions`, gets plain text back, and parses Qwen XML tool-call blocks in our own harness. Run the benchmark sequentially: concurrent τ³ simulations caused vLLM Metal OOMs and polluted the result cache with infrastructure failures.


In [3]:
tokenizer = cfg.load_tokenizer()
teacher_config = cfg.teacher_config_from_env(default_provider="vllm_raw", default_model=TEACHER_MODEL)

server_health = teacher_backends.teacher_server_health(teacher_config)
active_teacher_model = (server_health or {}).get("model", teacher_config.model_name)

print("Teacher provider:", teacher_config.provider)
print("Teacher server:", teacher_config.server_base_url)
print("Teacher request model:", teacher_config.request_model)
print("Active teacher model:", active_teacher_model)
print("Teacher temperature:", teacher_config.temperature)
print("Teacher top_p:", teacher_config.top_p)
print("Teacher top_k:", teacher_config.top_k)
print("Teacher max_new_tokens:", teacher_config.max_new_tokens)
print("Teacher runtime ready:", teacher_backends.teacher_runtime_is_configured(teacher_config))


Teacher provider: vllm_raw
Teacher server: http://127.0.0.1:8092
Teacher request model: mlx-community/Qwen3.5-35B-A3B-8bit
Active teacher model: mlx-community/Qwen3.5-35B-A3B-8bit
Teacher temperature: 0.0
Teacher top_p: 1.0
Teacher top_k: 0
Teacher max_new_tokens: 2048
Teacher runtime ready: True



## Load τ³-Bench Retail Runtime And Tools

The official τ³-Bench runtime owns the environment, hidden database, customer simulator, and reward calculation. We only provide the teacher agent wrapper.


In [4]:
retail_domain = tau_runtime.load_tau_bench_retail_domain(DATA_DIR, TAU_BENCH_REPO_REVISION)
tau_bench_runtime = retail_domain.runtime
retail_tasks = tau_bench_runtime.get_tasks("test")
retail_environment = retail_domain.environment
retail_tools = retail_domain.tools
retail_tool_schema_by_name = retail_domain.tool_schema_by_name

print("`tau2` package source checkout:", tau_bench_runtime.source_checkout)
print("Held-out retail tasks:", len(retail_tasks))
print("Retail tools:", len(retail_tools))
print("First test task id:", retail_tasks[0].id)
print("First five tools:", [tool["name"] for tool in retail_tools[:5]])


`tau2` package source checkout: /Users/kargarisaac/codes/personal/distillation-blogs/data/external/tau2-bench
Held-out retail tasks: 40
Retail tools: 16
First test task id: 5
First five tools: ['calculate', 'cancel_pending_order', 'exchange_delivered_order_items', 'find_user_id_by_name_zip', 'find_user_id_by_email']



## Smoke Check The User Simulator

This makes one customer-simulator call before we run the full eval. If this fails, the benchmark cannot produce a clean score.


In [5]:

user_simulator_check = retail_eval.check_tau_bench_retail_user_simulator(
    runtime=tau_bench_runtime,
    task=retail_tasks[0],
    user_simulator_model=TAU_BENCH_USER_SIMULATOR_LLM,
    user_simulator_args=TAU_BENCH_USER_SIMULATOR_ARGS,
)

print("User simulator check passed.")
print("Task id:", user_simulator_check["task_id"])
print("Model:", user_simulator_check["model"])
print("User-side tool count:", user_simulator_check["user_side_tool_count"])
if user_simulator_check["content"] and user_simulator_check["content"].strip():
    print("First simulated user message:")
    print(user_simulator_check["content"].strip())
if user_simulator_check["tool_calls"]:
    print("First simulated user tool calls:")
    print(json.dumps(user_simulator_check["tool_calls"], indent=2, ensure_ascii=False))


User simulator check passed.
Task id: 5
Model: openai/gpt-5.4-mini
User-side tool count: 0
First simulated user message:
Hi, I’d like to exchange a water bottle and a desk lamp I bought.


## Run Teacher Eval On Held-Out Retail Tasks

This is pass@1: one trajectory per task, no retry search. The output is cached task by task, so rerunning the cell resumes from the missing task IDs.

Notebook output is intentionally compact: a progress bar shows completed tasks, `valid/total`, and running accuracy. The noisy τ³/tau2 turn-by-turn logs are suppressed in the notebook. Full task traces are still saved locally, and when MLflow is enabled the same per-task details are logged as artifacts under one eval run.


In [ ]:
if not teacher_backends.teacher_runtime_is_configured(teacher_config):
    raise RuntimeError("Teacher server is not ready. Start the vLLM command above, then rerun this cell.")

TAU_BENCH_TEACHER_EVAL_LIMIT = len(retail_tasks)
TAU_BENCH_TEACHER_EVAL_MAX_STEPS = 100
TAU_BENCH_TEACHER_EVAL_MAX_ERRORS = 10
TAU_BENCH_TEACHER_EVAL_SEED = 42
TAU_BENCH_MLFLOW_ENABLED = True
TAU_BENCH_MLFLOW_EXPERIMENT_NAME = "distillation-blogs-tau3"
TAU_BENCH_MLFLOW_LOG_FULL_ARTIFACTS = True

user_simulator_slug = cfg.filename_slug(TAU_BENCH_USER_SIMULATOR_LLM)
teacher_slug = cfg.filename_slug(active_teacher_model)
teacher_eval_output_path = OUTPUT_DIR / f"{teacher_slug}_{teacher_config.provider}_tau3_bench_retail_test_official_teacher_eval_{user_simulator_slug}.json"
teacher_eval_trace_dir = OUTPUT_DIR / "local_traces" / f"{teacher_slug}_{teacher_config.provider}_tau3_bench_retail_test_official_teacher_eval_{user_simulator_slug}"
teacher_eval_mlflow_run_name = f"{teacher_slug}_{teacher_config.provider}_tau3_retail_teacher_eval_{user_simulator_slug}"
teacher_eval_mlflow_config = cfg.MlflowConfig(
    enabled=TAU_BENCH_MLFLOW_ENABLED,
    experiment_name=TAU_BENCH_MLFLOW_EXPERIMENT_NAME,
    log_full_artifacts=TAU_BENCH_MLFLOW_LOG_FULL_ARTIFACTS,
    log_spans=False,
)

teacher_eval_config = cfg.TauBenchRetailEvalConfig(
    dataset_revision=TAU_BENCH_REPO_REVISION,
    student_model_name=active_teacher_model,
    user_simulator_model=TAU_BENCH_USER_SIMULATOR_LLM,
    user_simulator_args=TAU_BENCH_USER_SIMULATOR_ARGS,
    max_steps=TAU_BENCH_TEACHER_EVAL_MAX_STEPS,
    max_errors=TAU_BENCH_TEACHER_EVAL_MAX_ERRORS,
    max_new_tokens=teacher_config.max_new_tokens,
    seed=TAU_BENCH_TEACHER_EVAL_SEED,
    model_role="teacher",
)

teacher_eval_task_objects = retail_tasks[:TAU_BENCH_TEACHER_EVAL_LIMIT]
teacher_eval_runner = retail_eval.TauBenchRetailTeacherEvalRunner(
    runtime=tau_bench_runtime,
    tokenizer=tokenizer,
    qwen_tools=retail_tools,
    tool_schema_by_name=retail_tool_schema_by_name,
    teacher_config=teacher_config,
    config=teacher_eval_config,
    trace_dir=teacher_eval_trace_dir,
)

print("Official τ³-Bench retail teacher eval")
print("Teacher model:", active_teacher_model)
print("Teacher backend:", teacher_config.provider, teacher_config.server_base_url)
print("User simulator model:", TAU_BENCH_USER_SIMULATOR_LLM)
print("User simulator args:", user_simulator.public_user_simulator_args(TAU_BENCH_USER_SIMULATOR_ARGS))
print("Held-out retail tasks:", len(teacher_eval_task_objects))
print("Task execution: sequential, one benchmark task at a time")
print("Max steps per task:", TAU_BENCH_TEACHER_EVAL_MAX_STEPS)
print("Max tool errors per task:", TAU_BENCH_TEACHER_EVAL_MAX_ERRORS)
print("Max new tokens per agent action:", teacher_config.max_new_tokens)
print("Output path:", teacher_eval_output_path)
print("Trace dir:", teacher_eval_trace_dir)
print("MLflow enabled:", teacher_eval_mlflow_config.enabled)
print("MLflow tracking URI:", teacher_eval_mlflow_config.tracking_uri)
print("MLflow experiment:", teacher_eval_mlflow_config.experiment_name)
print("MLflow full artifacts:", teacher_eval_mlflow_config.log_full_artifacts)
print("MLflow run name:", teacher_eval_mlflow_run_name)

teacher_eval_payload = retail_eval.run_tau_bench_retail_eval_tasks(
    task_objects=teacher_eval_task_objects,
    runner=teacher_eval_runner,
    output_path=teacher_eval_output_path,
    print_progress=True,
    show_progress_bar=True,
    quiet_tau2_console=True,
    mlflow_config=teacher_eval_mlflow_config,
    mlflow_run_name=teacher_eval_mlflow_run_name,
    mlflow_tags={"tau3.notebook": "02_teacher_eval", "tau3.runtime": teacher_config.provider},
)

teacher_eval_rows = teacher_eval_payload["task_results"]
teacher_eval_correct = sum(1 for row in teacher_eval_rows if row["is_success"])
teacher_eval_accuracy = teacher_eval_correct / len(teacher_eval_rows) if teacher_eval_rows else 0.0

print("\nTeacher official τ³-Bench retail pass@1 accuracy:", round(teacher_eval_accuracy, 4))
print(f"Correct: {teacher_eval_correct}/{len(teacher_eval_rows)}")
print("Saved aggregate results to:", teacher_eval_output_path)
print("Saved per-task traces to:", teacher_eval_trace_dir)
if teacher_eval_mlflow_config.enabled:
    print("MLflow run:", teacher_eval_mlflow_run_name)
    print("MLflow experiment:", teacher_eval_mlflow_config.experiment_name)

failure_counter = Counter(
    row["termination_reason"] if not row.get("error") else f"error:{row['error']['type']}"
    for row in teacher_eval_rows
    if not row["is_success"]
)
print("\nFailure/termination summary:")
for name, count in failure_counter.most_common():
    print(f"  {name}: {count}")